# Конспект. Модуль 5: GBM в деталях — инженерные «ручки» алгоритма

## 1. Зачем это нужно и как это связано с предыдущими модулями

В конце Модуля 4 мы прошли две итерации бустинга вручную и увидели: train MSE монотонно убывала — `106.56 -> 39.89 -> 13.85`. Возникает естественный вопрос, который мы тогда сознательно отложили: **что будет, если продолжать добавлять деревья бесконечно — 50, 100, 1000 итераций?** Train-ошибка продолжит падать (мы уже видели в разделе 5.1 Модуля 4, что при `η=1` и достаточно «сильном» дереве можно вообще довести её до нуля за одну итерацию) — но приведёт ли это к лучшей модели на **новых** данных?

Ответ — нет, не всегда, и именно с этим вопросом справляется Модуль 5. Здесь мы разбираем три инженерных механизма, которые превращают «теоретически работающий» алгоритм из Модуля 3–4 в **реальный, контролируемый инструмент**: `learning_rate` (shrinkage), `subsample` (стохастичность) и **раннюю остановку**. Все три параметра, по сути, отвечают на один и тот же вопрос с разных сторон: как **не дать** бустингу «слишком хорошо» выучить обучающую выборку.

## 2. Shrinkage (learning rate `η`)

### 2.1. Напоминание и переформулировка

В Модуле 3–4 мы уже использовали `η` в правиле обновления:

In [ ]:
F_m(x) = F_{m-1}(x) + η · h_m(x)

До сих пор мы относились к `η` просто как к «размеру шага» из обычного градиентного спуска (Модуль 3, раздел 3). Сейчас разберём **зачем вообще намеренно брать `η < 1`**, если, казалось бы, дерево `h_m` уже честно посчитано как лучшее приближение антиградиента — почему не довериться ему полностью?

### 2.2. Интуиция: почему не доверять дереву на 100%

Дерево `h_m(x)` обучается **не на истинном антиградиенте функции потерь**, а на его **оценке**, посчитанной по конкретной, конечной и зашумлённой обучающей выборке. Эта оценка сама по себе — случайная величина: возьми чуть другую выборку (или, как мы увидим в разделе 3, чуть другое случайное подмножество строк) — дерево `h_m` получится немного другим.

Если мы полностью доверяем каждому такому шуму (`η=1`), мы **полностью встраиваем** эту случайную ошибку оценки в модель на каждой итерации. Если же мы берём **частичный** шаг (`η` мало, например 0.05), мы говорим: «это дерево, скорее всего, указывает в правильном направлении, но не будем слепо доверять его точной величине — сделаем маленький, осторожный шаг, и посмотрим на следующей итерации, что покажет новое дерево, обученное уже на обновлённых остатках». Много маленьких, взаимно уточняющих шагов статистически надёжнее, чем несколько крупных и самоуверенных.

### 2.3. Формальная иллюстрация trade-off `η` и числа итераций

Сделаем **упрощающее (идеализированное) допущение**, чтобы вывести чистую формулу: представим, что на каждой итерации дерево `h_m` **идеально** повторяет весь вектор текущих остатков (`h_m = r^(m-1)` точно, без ошибки аппроксимации — реальное дерево, конечно, так не работает, особенно если это слабое неглубокое дерево из Модуля 4, но допущение полезно, чтобы изолированно увидеть **чистый эффект** `η` на скорость сходимости, без побочных эффектов качества конкретных сплитов).

При таком допущении:

In [ ]:
r^(m) = y - F_m = y - F_{m-1} - η·h_m = r^(m-1) - η·r^(m-1) = (1-η)·r^(m-1)

По индукции:

In [ ]:
r^(m) = (1-η)^m · r^(0)

**Это чистая геометрическая прогрессия.** При `η=1` остаток обнуляется за одну итерацию (согласуется с тем, что мы обсуждали в Модуле 4, раздел 5.1 — переобучение за один шаг). При меньших `η` остаток убывает медленнее, но **предсказуемо**.

**Сколько итераций нужно, чтобы уменьшить остаток до заданной доли `f` от начального значения** (например, `f=0.01` — уменьшить в 100 раз)? Решаем `(1-η)^m = f` относительно `m`:

In [ ]:
m = ln(f) / ln(1-η)

Посчитаем для `f=0.01` при разных `η`:

| η | Итераций для уменьшения остатка в 100 раз | Во сколько раз больше, чем при η=0.1 |
|---|---|---|
| 0.5 | ≈ 7 | 0.16× |
| 0.1 | ≈ 44 | 1× (база) |
| 0.05 | ≈ 90 | 2.05× |
| 0.01 | ≈ 458 | 10.4× |

**Ключевое наблюдение:** при достаточно малых `η` (реалистичный практический диапазон — примерно `0.01`–`0.3`) выполняется приближение `ln(1-η) ≈ -η` (стандартное разложение логарифма при малом аргументе — если знакомо из матанализа, это первый член ряда Тейлора для `ln(1-x)`), поэтому:

In [ ]:
m ≈ -ln(f) / η

то есть **число нужных итераций примерно обратно пропорционально `η`** — уменьшили `η` в 10 раз, нужно примерно в 10 раз больше итераций, что мы и видим в таблице (`0.01` даёт почти ровно в 10.4 раза больше итераций, чем `0.1`). Это и есть формальное обоснование практического правила, часто звучащего на собеседованиях без вывода: **«если уменьшаете learning_rate в N раз, стоит примерно в N раз увеличить n_estimators, чтобы модель успела сойтись»**.

**Важная честная оговорка:** это упрощённая идеализированная модель (дерево = точный повтор остатка). В реальности каждое дерево — слабый learner, аппроксимирующий остаток лишь частично, и разные `η` приводят к тому, что на каждой итерации строятся **разные** деревья (с разными сплитами, как мы наблюдали в Модуле 4, где второе дерево нашло другой порог, а не просто уточнило первое). Поэтому точной обратной пропорциональности в реальности не будет, но **направление и порядок величины** — увеличивать `n_estimators` примерно пропорционально уменьшению `η` — подтверждается эмпирически (это отдельно показал Фридман в своей оригинальной статье 2001 года "Greedy Function Approximation: A Gradient Boosting Machine") и является стандартной практикой тюнинга.

### 2.4. Почему меньший `η` даёт лучшее **обобщение**, а не просто «медленнее сходится»

Мало просто сказать «меньший `η` требует больше итераций» — важно понимать, **почему** маленький `η` + много итераций часто даёт модель, которая лучше работает на новых данных, чем большой `η` + мало итераций, даже если итоговый train loss у обеих моделей похож.

Интуиция (эмпирически подтверждённая в статье Фридмана, строгого доказательства «на пальцах» не существует, но логика такая): при большом `η` каждое дерево вносит **крупный, самоуверенный** вклад в модель — если это дерево слегка переоценило сигнал в остатках (out of шума), эта ошибка сразу входит в модель почти в полном объёме и **зафиксирована**. При маленьком `η` вклад каждого отдельного дерева невелик, и если следующее дерево увидит, что предыдущий шаг был чуть неверным (остаток скорректируется в другую сторону), ошибка **частично компенсируется** на следующих итерациях. Таким образом, `η` действует как своего рода **регуляризатор** — снижает эффективную «уверенность» каждого шага, распределяя финальное решение по многим согласованным между собой маленьким шагам, а не по нескольким крупным, каждый из которых рискует переобучиться на локальном шуме своей итерации.

## 3. Stochastic Gradient Boosting: `subsample`

### 3.1. Идея

До сих пор каждое дерево `h_m` обучалось на псевдо-остатках **всех** `N` обучающих объектов. **Stochastic Gradient Boosting** (Фридман, отдельная статья 1999 года) предлагает: на каждой итерации `m` случайным образом выбирать **подмножество** объектов (без возвращения, в отличие от bootstrap в Модуле 2 — обычно это называется `subsample`, доля от 0 до 1, например `0.5` — «использовать 50% случайно выбранных строк»), считать псевдо-остатки только для них и обучать `h_m` только на этой подвыборке.

### 3.2. Почему это помогает — двойной эффект

**Эффект 1 — скорость.** Меньше строк на каждой итерации -> быстрее строится каждое дерево (особенно заметно на больших датасетах, вроде вашего IEEE-CIS с ~590K строк).

**Эффект 2 — регуляризация через декорреляцию последовательных шагов.** Это прямая параллель с тем, что мы разбирали в Модуле 2 про Random Forest: там случайная подвыборка строк и признаков декоррелировала **параллельные** деревья, снижая variance ансамбля через усреднение. Здесь тот же приём случайного сэмплирования применяется к **последовательным** деревьям — каждое `h_m` видит немного другой срез данных, поэтому не может идеально «переобучиться» под весь набор остатков целиком (просто потому, что не видит его целиком) — это добавляет полезный шум, который **мешает** модели слишком точно подогнаться под конкретные обучающие точки на каждом отдельном шаге.

**Практическое наблюдение (эмпирика Фридмана):** типичные значения `subsample` в диапазоне `0.5–0.8` часто **одновременно** ускоряют обучение **и** улучшают итоговое качество на валидации — довольно редкий случай, когда регуляризация «бесплатна», а не является строгим компромиссом (обычно регуляризация чем-то жертвует ради устойчивости — здесь же выигрыш почти без потерь, что делает `subsample` одним из самых «безопасных» гиперпараметров для включения по умолчанию).

**Отдельно от `subsample` по строкам** существует аналогичный приём для **признаков** — на каждой итерации (или для каждого разбиения) рассматривать случайное подмножество столбцов (`max_features` в терминах Модуля 2, в градиентном бустинге в sklearn — тот же параметр `max_features`, в LightGBM это будет `feature_fraction`, забегая вперёд к Модулю 7). Механика та же — дополнительная декорреляция и регуляризация через ограничение обзора модели на каждом шаге.

## 4. Почему бустинг может переобучаться от числа итераций, а Random Forest — почти нет

Это, вероятно, самый важный концептуальный вопрос модуля — прямое продолжение таблицы сравнения из Модуля 4 (раздел 5.3), но теперь применительно конкретно к **числу базовых моделей** (`n_estimators` / `B`).

### 4.1. Формальное напоминание из Модуля 2

Для Random Forest мы вывели:

In [ ]:
Var(среднее B деревьев) = ρ·σ² + (1-ρ)·σ²/B

Здесь `B` **не входит** ни в какое слагаемое, которое **растёт** — увеличение `B` может только **уменьшать** второе слагаемое (стремится к 0 при `B->∞`), первое же слагаемое (`ρσ²`) от `B` вообще не зависит. **Формула физически не позволяет** variance расти с ростом `B` — значит, ожидаемая ошибка на новых данных не может ухудшиться от добавления новых деревьев (в пределе — выходит на плато).

### 4.2. Почему в градиентном бустинге всё иначе

В градиентном бустинге число итераций `M` — это **не число независимых оценок одной и той же модели**, а число **последовательных добавочных слагаемых**, из которых складывается итоговая функция:

In [ ]:
F_M(x) = F_0(x) + η·h_1(x) + η·h_2(x) + ... + η·h_M(x)

Каждое новое `h_m` — это **дополнительная степень свободы**, добавляющая функции `F_M` больше «изгибов», больше способности различать тонкие детали в данных. С ростом `M` эффективная сложность (способность модели описывать сложные, нелинейные зависимости) **монотонно растёт** — совершенно так же, как росла сложность одиночного дерева при увеличении `max_depth` в Модуле 1.

**Прямая аналогия с Модулем 1:** там `max_depth` управлял тем, сколько «вопросов» может задать одно дерево прежде, чем дать ответ — больше глубина, больше bias снижается, но variance растёт. В градиентном бустинге роль «глубины» для **всего ансамбля** играет именно `M` (число итераций) — большее `M` снижает bias (модель точнее приближает обучающие данные), но за счёт роста variance всего ансамбля (модель становится чувствительнее к шуму конкретной обучающей выборки).

### 4.3. Что происходит содержательно на поздних итерациях

На первых итерациях бустинг ловит **крупный, систематический сигнал** в данных (в нашем примере Модуля 4 — грубое разделение на 2 группы по `x≤7.5`, затем более тонкое разделение внутри одной из групп). Чем дальше, тем «легкий» сигнал уже выловлен предыдущими деревьями, и **остатки, на которых обучаются последующие деревья, всё больше состоят из шума**, а не из настоящей закономерности.

Проблема в том, что дерево **не умеет отличить** «маленький, но настоящий сигнал» от «случайного шума конкретной обучающей выборки» — оно **всегда** найдёт какое-то разбиение с ненулевым падением impurity (Модуль 1), даже если оно фактически подгоняется под случайные флуктуации. Продолжая добавлять деревья после того, как «легкий» сигнал исчерпан, модель начинает **активно** описывать шум — это классическое переобучение, но, в отличие от Random Forest, оно **не гасится** механизмом самого алгоритма (усреднением) — здесь каждое новое дерево реально **меняет** итоговую функцию `F_M`, а не просто добавляет ещё одну независимую оценку той же самой сложности функции.

**Формулировка для собеседования (запомнить дословно):** *«В Random Forest число деревьев `B` управляет только variance усреднения ФИКСИРОВАННОЙ по сложности модели — рост `B` безопасен. В градиентном бустинге число итераций `M` управляет ЭФФЕКТИВНОЙ СЛОЖНОСТЬЮ самой модели — рост `M` снижает bias, но неограниченно увеличивает variance, поэтому `M`, как и `max_depth` одиночного дерева, требует контроля и имеет оптимальную точку, а не может расти бесконечно без вреда»*.

## 5. Ранняя остановка (Early Stopping)

### 5.1. Механизм

Раз мы установили, что у `M` есть оптимальная точка (не слишком мало — недообучение, не слишком много — переобучение), нужен **практический способ** её найти, не перебирая все значения `M` вручную через отдельные полные обучения (это было бы то же самое, что `GridSearchCV` по одному параметру, но крайне неэффективно — обучать модель заново для каждого кандидата `M`, когда на самом деле промежуточные модели `F_1, F_2, ..., F_M` уже вычисляются последовательно в рамках **одного** обучения).

**Идея ранней остановки:** мониторить качество на **отдельной валидационной выборке** **после каждой добавленной итерации** (это дёшево — модель `F_m` уже посчитана, остаётся просто применить её к валидационным данным) и остановить обучение, как только валидационная метрика перестаёт улучшаться.

**Алгоритм:**

In [ ]:
best_val_loss = +∞
patience_counter = 0
patience = N  (например, 20 - гиперпараметр допустимого "терпения")

для m = 1, ..., M_max:
    построить F_m (одна итерация бустинга)
    посчитать val_loss на валидационной выборке через F_m

    если val_loss < best_val_loss:
        best_val_loss = val_loss
        best_m = m
        patience_counter = 0
    иначе:
        patience_counter += 1

    если patience_counter >= patience:
        остановить обучение
        вернуть модель F_(best_m)  # не последнюю, а лучшую по валидации!

**Важная деталь, которую часто упускают:** при остановке нужно **вернуть модель на итерации с лучшим `val_loss`**, а не на той итерации, где сработал критерий остановки — эти два момента, как правило, различаются на величину `patience`, потому что мы даём модели `patience` дополнительных попыток «одуматься» прежде, чем сдаться.

### 5.2. Как это выглядит на графике (learning curve)

Типичная картина: train loss **монотонно убывает** на всём протяжении обучения (мы теоретически обосновали это в разделе 4.2 — каждая новая итерация может только уменьшить или не изменить ошибку **на обучающих данных**, так как это прямая цель оптимизации). Val loss сначала **тоже убывает** (модель ловит настоящий сигнал, который обобщается на новые данные), достигает **минимума**, а затем начинает **расти** (модель начинает описывать шум конкретной обучающей выборки, что вредит обобщению — раздел 4.3).

In [ ]:
Loss
 │
 │  train loss ────────────────────────╲___________________
 │                                       ╲___________________
 │
 │  val loss   ╲                                    ___________
 │              ╲___________                    ___╱
 │                          ╲__________     ___╱
 │                                     ╲___╱  <- минимум val loss,
 │                                              точка ранней остановки
 └────────────────────────────────────────────────────────► итерация m

**Практическое правило чтения графика:** искать глазами точку, где val-кривая **перестаёт снижаться и начинает расти** (либо выходит на устойчивое плато без улучшений). Именно там — оптимальное `M`.

## 6. Практика: код

### 6.1. Построение learning curve вручную через `staged_predict`

sklearn предоставляет удобный метод `staged_predict`/`staged_predict_proba`, который последовательно выдаёт предсказания модели **после каждой добавленной итерации** — не нужно обучать `M` разных моделей отдельно, всё считается за один `.fit()`.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

X, y = make_classification(n_samples=3000, n_features=20, n_informative=10,
                            flip_y=0.05, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

gbc = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=3, random_state=42
)
gbc.fit(X_train, y_train)

train_losses, val_losses = [], []
for y_pred_train, y_pred_val in zip(
    gbc.staged_predict_proba(X_train), gbc.staged_predict_proba(X_val)
):
    train_losses.append(log_loss(y_train, y_pred_train))
    val_losses.append(log_loss(y_val, y_pred_val))

plt.plot(train_losses, label="train logloss")
plt.plot(val_losses, label="val logloss")
plt.axvline(np.argmin(val_losses), color="r", linestyle="--",
            label=f"оптимум: итерация {np.argmin(val_losses)}")
plt.xlabel("Итерация (номер дерева)")
plt.ylabel("LogLoss")
plt.legend()
plt.show()

print(f"Лучшая итерация по валидации: {np.argmin(val_losses)}")
print(f"Итоговое число итераций (n_estimators): {gbc.n_estimators}")

**Что искать:** train-кривая должна монотонно убывать почти до самого конца, val-кривая — образовать характерную «ложбину» с минимумом где-то заметно раньше `n_estimators=300` (если данные достаточно шумные — `flip_y=0.05` специально добавляет шум в метки для наглядной демонстрации переобучения).

### 6.2. Встроенная ранняя остановка sklearn

In [ ]:
gbc_early = GradientBoostingClassifier(
    n_estimators=1000,       # намеренно большое верхнее ограничение
    learning_rate=0.1,
    max_depth=3,
    validation_fraction=0.2,  # доля train, отложенная под внутреннюю валидацию
    n_iter_no_change=20,      # "patience" - сколько итераций без улучшения ждать
    tol=1e-4,                 # минимальный порог, который считается "улучшением"
    random_state=42
)
gbc_early.fit(X_train, y_train)

print(f"Реально построено деревьев: {gbc_early.n_estimators_}")

`n_estimators_` (с подчёркиванием в конце — атрибут **обученной** модели) покажет, на какой итерации обучение реально остановилось — если оно меньше заданного `n_estimators=1000`, значит сработала ранняя остановка.

### 6.3. Демонстрация trade-off `learning_rate` vs `n_estimators`

In [ ]:
configs = [
    {"learning_rate": 0.3, "n_estimators": 50},
    {"learning_rate": 0.1, "n_estimators": 150},
    {"learning_rate": 0.03, "n_estimators": 500},
    {"learning_rate": 0.01, "n_estimators": 1500},
]

for cfg in configs:
    model = GradientBoostingClassifier(max_depth=3, random_state=42, **cfg)
    model.fit(X_train, y_train)
    val_loss = log_loss(y_val, model.predict_proba(X_val))
    print(f"lr={cfg['learning_rate']:<5} n_estimators={cfg['n_estimators']:<5} "
          f"-> val logloss = {val_loss:.4f}")

Обратите внимание на примерное соответствие `learning_rate × n_estimators` — каждая следующая конфигурация уменьшает `learning_rate` примерно втрое и увеличивает `n_estimators` примерно в 3 раза, в духе вывода раздела 2.3. Часто (хотя не гарантированно) конфигурации с **меньшим** `learning_rate` и **большим** `n_estimators` дают лучший (более низкий) `val_loss`, хотя и требуют больше времени на обучение — классический trade-off «качество против времени вычислений».

### 6.4. Эффект `subsample`

In [ ]:
for ss in [1.0, 0.8, 0.5, 0.3]:
    model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3,
        subsample=ss, random_state=42
    )
    model.fit(X_train, y_train)
    val_loss = log_loss(y_val, model.predict_proba(X_val))
    print(f"subsample={ss} -> val logloss = {val_loss:.4f}")

## 7. Связь с вашим проектом FraudGuard

В коде `train.py` вашего проекта FraudGuard (День 2) уже используются конкретные значения этих параметров:

In [ ]:
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    class_weight='balanced',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    ...
)

Теперь вы можете осознанно прочитать эту конфигурацию: `learning_rate=0.05` — довольно небольшой шаг (в реалистичном диапазоне из раздела 2.3), что требует **достаточно большого** `n_estimators=500`, чтобы модель успела сойтись — именно такое сочетание мы и обосновали формально в этом модуле. В плане Дня 3 вашего проекта также заложена идея сохранения `pr_auc` на holdout — по сути, это именно та валидационная метрика, по которой в перспективе стоит подбирать оптимальное число итераций через раннюю остановку (в LightGBM — параметры `early_stopping_rounds` и `eval_set`, подробно разберём в Модуле 7).

## 8. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Зачем нужен `learning_rate < 1`, если дерево и так обучено на «правильном» антиградиенте? | Дерево — лишь оценка антиградиента по шумной конечной выборке; частичный шаг снижает риск переноса шума одной итерации в модель целиком, действует как регуляризатор |
| Как связаны `learning_rate` и `n_estimators`? | Обратная зависимость (примерно пропорциональная в реалистичном диапазоне малых `η`) — уменьшение `η` в `N` раз требует примерно в `N` раз больше итераций для сопоставимой сходимости |
| Почему `subsample < 1` может одновременно ускорить обучение и улучшить качество? | Меньше строк на итерацию — быстрее; дополнительная случайность декоррелирует последовательные деревья, действуя как регуляризатор, аналогично bootstrap в Random Forest, но применительно к последовательным, а не параллельным моделям |
| Почему увеличение `n_estimators` в градиентном бустинге может ухудшить качество на тесте, а в Random Forest — почти никогда? | В Random Forest `B` управляет только variance усреднения моделей фиксированной сложности (формула `ρσ²+(1-ρ)σ²/B` не растёт с `B`). В бустинге `M` определяет эффективную сложность самой аддитивной функции — рост `M` снижает bias, но неограниченно увеличивает variance, аналогично росту глубины одиночного дерева |
| Как реализована ранняя остановка технически? | Мониторинг метрики на валидации после каждой добавленной итерации, счётчик итераций без улучшения (`patience`), остановка при превышении порога, возврат модели с **лучшей**, а не последней итерацией |

## 9. Чек-поинт — попробуйте ответить без подсказок

1. Если уменьшить `learning_rate` в 10 раз, что нужно сделать с `n_estimators`?
2. Как по графику learning curve понять, что пора остановиться?
3. Почему `subsample < 1` часто улучшает **и** скорость, **и** качество одновременно — в чём тут «бесплатный обед»?
4. Почему увеличение `n_estimators` в градиентном бустинге, в отличие от Random Forest, может ухудшить качество на новых данных?
5. Что конкретно происходит содержательно с остатками (`pseudo-residuals`) на поздних итерациях бустинга, из-за чего дальнейшее обучение начинает вредить обобщению?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Нужно примерно в 10 раз увеличить `n_estimators`, чтобы модель успела сойтись к сопоставимому уровню общей коррекции — это следует из идеализированного вывода `m ≈ -ln(f)/η` (при малых `η` число нужных итераций обратно пропорционально `η`), подтверждённого также эмпирически в оригинальной статье Фридмана.

2. Нужно смотреть на **валидационную** кривую (не train — она почти всегда монотонно убывает и не покажет переобучения): точка, где val loss перестаёт снижаться и начинает расти (или выходит на устойчивое плато) — это точка оптимального числа итераций; именно она (а не последняя, если продолжить обучение дальше) должна использоваться как финальная модель.

3. Стохастическое сэмплирование строк даёт сразу два независимых выигрыша: обучение на меньшем количестве строк физически быстрее (прямой выигрыш по времени), и одновременно случайная декорреляция последовательных деревьев действует как регуляризатор, снижая переобучение (аналогично тому, как bootstrap снижает корреляцию деревьев в Random Forest, только здесь — между последовательными шагами, а не параллельными моделями). Обычно регуляризация требует жертвовать чем-то (например, bias ради variance), но здесь оба эффекта (скорость и качество) чаще всего совпадают по направлению — редкий случай выигрыша без явного компромисса.

4. В Random Forest число деревьев `B` управляет только тем, сколько **независимых оценок одной и той же по сложности** модели усредняется — variance усреднения не может расти с `B` (формула `ρσ²+(1-ρ)σ²/B`, где `B` входит только со знаком минус). В градиентном бустинге число итераций `M` определяет **эффективную сложность самой модели** — каждое новое дерево добавляет ансамблю новую степень свободы, снижая bias, но при этом неограниченно увеличивая variance всей модели, точно так же, как рост `max_depth` увеличивал variance одиночного дерева в Модуле 1.

5. На первых итерациях бустинг ловит крупный, систематический сигнал (объясняет большую часть отклонения `y` от текущего предсказания). По мере того как этот сигнал «выловлен», остатки, на которых обучаются последующие деревья, всё сильнее состоят из случайного шума конкретной обучающей выборки, а не из настоящей закономерности. Так как дерево не умеет отличить слабый истинный сигнал от шума (оно всегда найдёт какое-то ненулевое падение impurity при разбиении, Модуль 1), продолжение обучения на этой стадии заставляет модель активно **подстраиваться под шум**, что улучшает train-метрику, но ухудшает обобщающую способность на новых данных.

</details>